# Interactive Swap Pricing & Risk Workbench (`legend-lite`)

This notebook demonstrates how to supply:
1. **Benchmark Par Swap Quotes** for both the **OIS Discount Curve** (`USD-SOFR-OIS`) and the **3M Projection Curve** (`USD-SOFR-3M`),
2. **A Specific Swap Contract** (`IRSwapFixedFloat` with explicit floating coupons across two curves, or `IRSwapFixedFloatApprox` on one curve),
3. **Execution Engine Choice** (`PureBasisEngine` producing Pure & SQL and executing in DuckDB vs. `PythonEngineExpr` evaluating `expr.Expr` DAGs locally in Python),
4. **Shared Risk Definitions** (`FirstOrderAnalyticRisk` via symbolic differentiation + Jacobian chain rule vs. `FirstOrderNumericalRisk` via naive per-quote bump, curve refit & portfolio reprice).

In [ ]:
import sys
from pathlib import Path

# Ensure python/ is on sys.path when running directly from python/pricing/demo/
py_root = str(Path.cwd().resolve().parents[1]) if Path.cwd().name == 'demo' else str(Path.cwd() / 'python')
if py_root not in sys.path:
    sys.path.insert(0, py_root)

from pricing.demo.irs_workbench import IRSWorkbench
from pricing.engines import PureBasisEngine, PythonEngineExpr
from pricing.risk import FirstOrderAnalyticRisk, FirstOrderNumericalRisk

# 1. Initialize the workbench with custom benchmark quotes (in bps) for 1Y, 2Y, 3Y, 5Y, 7Y, 10Y
wb = IRSWorkbench(
    ois_quotes=[300.0, 320.0, 335.0, 355.0, 368.0, 380.0],  # USD-SOFR-OIS (Discount Curve)
    irs_quotes=[315.0, 338.0, 355.0, 378.0, 392.0, 405.0],  # USD-SOFR-3M  (Projection Curve)
    engine='pure_duckdb',                                    # or 'python'
)

# 2. Specify a swap to price and print its full valuation & risk report
frames = wb.price_swap(
    symbol='DEAL_5Y_FWD',
    book='CLIENT_DESK',
    notional=25_000_000,
    fixed_rate_bps=365.0,
    float_spread_bps=5.0,
    start=0.5,
    tenor_years=5.0,
    side='RECEIVER',
    two_curve=True,
)

## 2. Compare Engines (`PythonEngineExpr` vs `PureBasisEngine`) & Risk Methods (`FirstOrderAnalyticRisk` vs `FirstOrderNumericalRisk`)
You can evaluate the exact same swap portfolio using either engine and either risk definition, and inspect the Pure queries and compiled SQL:

In [ ]:
py_eng = PythonEngineExpr()
sql_eng = wb.engine  # PureBasisEngine

analytic_py = py_eng.quote_risk(wb.swaps, wb.fitter, wb.run.fit, risk_method=FirstOrderAnalyticRisk)
analytic_sql = sql_eng.quote_risk(wb.swaps, wb.fitter, wb.run.fit, risk_method=FirstOrderAnalyticRisk)
bump_grind_sql = sql_eng.quote_risk(wb.swaps, wb.fitter, wb.run.fit, risk_method=FirstOrderNumericalRisk, method='forward')
bump_grind_central = py_eng.quote_risk(wb.swaps, wb.fitter, wb.run.fit, risk_method=FirstOrderNumericalRisk, method='central')

print(f"{'Quote ID':<10} {'SQL Analytic':>14} {'Py Analytic':>14} {'Bump&Refit (1bp)':>18} {'Bump&Refit (Central)':>22}")
for q in wb.fitter.quote_specs:
    k = ('CLIENT_DESK', q.quote_id)
    print(f"{q.quote_id:<10} {analytic_sql.get(k, 0.0):>14,.2f} {analytic_py.get(k, 0.0):>14,.2f} "
          f"{bump_grind_sql.get(k, 0.0):>18,.2f} {bump_grind_central.get(k, 0.0):>22,.2f}")

print('\nCompiled DuckDB SQL for Quote Risk (Jacobian Chain Rule):')
print(wb.plan_sql('duckdb')['quote_risk'])

## 3. Open the Live DataCube Dashboard
Calling `wb.show()` displays the 4-sheet DataCube `ll.Page` widget right under the cell (or in a browser tab if `inline=False`). Any subsequent calls to `wb.bump_quote(...)`, `wb.set_quote(...)`, or `wb.price_swap(...)` automatically update the open dashboard in place.

In [ ]:
wb.show()

## 4. Modify Quotes or Swap Parameters Live
Try bumping the 5Y IRS benchmark quote by `+10 bps` or adding a second 1-curve OIS hedge swap:

In [ ]:
# Bump the 5Y IRS projection quote by +10 bps and add a 5Y Pay-Fixed OIS hedge:
wb.bump_quote('IRS_5Y', +10.0, verbose=False)
wb.add_swap(
    symbol='HEDGE_OIS_5Y',
    book='CLIENT_DESK',
    notional=25_000_000,
    fixed_rate_bps=355.0,
    start=0.0,
    tenor_years=5.0,
    side='PAYER',
    two_curve=False,
)

## 5. Interactive `ipywidgets` Control Panel + DataCube
In JupyterLab / VS Code with `ipywidgets` and `anywidget` installed, `wb.widget()` renders sliders, engine selector, and quote boxes directly above the live DataCube page:

In [ ]:
wb.widget()